# Вероятность выбора семантического маршрута LLM

## Аннотация

В этой работе я исследую задачу бинарной маршрутизации диалога: по истории диалога и новому сообщению необходимо определить, продолжает ли пользователь текущую задачу (`CONTINUE`) или начинает новую самостоятельную задачу (`NEW`).

Исходная идея состояла в том, чтобы использовать `logprobs` языковой модели как относительную оценку вероятности выбора намерения. По мере экспериментов оказалось, что результат зависит не только от семантики сообщения, но и от способа представления вариантов в prompt. Поэтому дальнейшие эксперименты были построены как последовательная проверка конкретных гипотез: влияние буквенных меток, порядка кандидатов, независимой оценки кандидатов, специализированного reranker, прямого structured-выбора, явного reasoning и размера модели.

Этот notebook оформлен как журнал исследования. Каждый этап имеет одну и ту же структуру:

1. **Проблема** — что стало непонятно после предыдущего эксперимента.
2. **Гипотеза** — что именно я хочу проверить.
3. **Эксперимент** — что меняется и что остаётся фиксированным.
4. **Результат** — значения берутся из сохранённых артефактов эксперимента.
5. **Вывод** — что можно утверждать по полученным данным.
6. **Следующая гипотеза** — почему появился следующий эксперимент.


## Постановка задачи

Для каждого примера заданы:

- история текущего диалога;
- новое сообщение пользователя;
- эталонный семантический маршрут.

Используются два маршрута:

- `CONTINUE` — новое сообщение продолжает текущую пользовательскую задачу;
- `NEW` — новое сообщение вводит новую независимую задачу.

На первых этапах используется диагностический датасет из 30 случаев. Он сбалансирован по двум классам и нужен прежде всего для анализа поведения метода. Позже я отдельно перехожу к Dataset V3, потому что многократное использование одних и тех же 30 примеров делает их непригодными как окончательный независимый benchmark.


## Служебная ячейка: чтение сохранённых результатов

Я не переношу числа в notebook вручную там, где их можно получить непосредственно из сохранённых CSV/JSON. Это уменьшает риск расхождения между текстом и экспериментальными артефактами.


In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

def read_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def read_csv(relative_path):
    with (ROOT / relative_path).open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))

def accuracy_from_csv(relative_path):
    rows = read_csv(relative_path)
    valid = [row for row in rows if not row.get("error")]
    correct = sum(row["correct"].lower() == "true" for row in valid)
    return {
        "correct": correct,
        "total": len(valid),
        "accuracy": correct / len(valid) if valid else None,
    }

def percent(value):
    return f"{100 * value:.2f}%" if value is not None else "—"


## 1. Исходный эксперимент: выбор между A и B

### Проблема

Первоначально я хотел получить относительную вероятность выбора намерения непосредственно из `logprobs`. Для этого модель должна была ответить одним из двух токенов — `A` или `B`.

Но сами буквы не имеют нужной семантики. Поэтому возникает вопрос: не зависит ли решение от того, какой смысл назначен букве и в каком месте prompt находится её определение?

### Гипотеза

Если модель действительно ориентируется прежде всего на содержание маршрута, перестановка семантики `A/B` и положения определения `A` не должна радикально менять качество.

### Эксперимент

Я зафиксировал четыре условия:

| Условие | Семантика A | Семантика B | Позиция A |
|---|---|---|---|
| E1 | CONTINUE | NEW | первая |
| E2 | NEW | CONTINUE | первая |
| E3 | CONTINUE | NEW | вторая |
| E4 | NEW | CONTINUE | вторая |

Для каждого случая модель могла сгенерировать только `A` или `B`; решение определялось по их `logprobs`.


In [ ]:
prompt_runs = {
    "E1": "results/prompt_conditions/legacy/run_20260903_120522_E1.csv",
    "E2": "results/prompt_conditions/legacy/run_20260903_141001_E2.csv",
    "E3": "results/prompt_conditions/legacy/run_20260903_142756_E3.csv",
    "E4": "results/prompt_conditions/legacy/run_20260903_142152_E4.csv",
}

prompt_summary = {
    condition: accuracy_from_csv(path)
    for condition, path in prompt_runs.items()
}
prompt_summary


### Наблюдение и вывод

Результаты заметно менялись между условиями E1–E4; в сохранённых экспериментах разброс accuracy составлял примерно от 50% до 93.33%.

Это означает, что `logprob(A)` и `logprob(B)` нельзя интерпретировать как нейтральную оценку двух семантических намерений. На решение влияет способ кодирования классов и их расположение в prompt.

### Следующая гипотеза

Возможно, проблема в самих бессмысленных буквах `A/B`. Если вместо них использовать настоящие названия маршрутов `CONTINUE` и `NEW`, чувствительность к представлению должна уменьшиться.


## 2. Семантические названия маршрутов и порядок кандидатов

### Проблема

После A/B-эксперимента нужно отделить влияние буквенных токенов от более общего эффекта порядка кандидатов.

### Гипотеза

Если использовать семантические названия `CONTINUE` и `NEW`, модель будет меньше зависеть от поверхностного представления вариантов.

### Эксперимент

Оба маршрута показывались модели одновременно, но уже под собственными именами и с описаниями. Затем порядок `CONTINUE → NEW` менялся на `NEW → CONTINUE`.

### Результат

В исторических запусках accuracy составила примерно 90.00% и 73.33%, а semantic agreement между двумя порядками — 21/30 = 70%. Девять решений изменились только из-за перестановки кандидатов.

### Вывод

Замена A/B на семантические названия улучшила интерпретируемость, но не устранила зависимость от порядка. Значит, проблема шире, чем token prior для букв A и B: само совместное предъявление кандидатов влияет на решение.

### Следующая гипотеза

Чтобы убрать конкуренцию кандидатов внутри одного prompt, каждый маршрут нужно оценивать отдельно, не показывая модели альтернативный маршрут.


## 3. Independent candidate scoring

### Проблема

В listwise-подходе оба кандидата находятся в одном контексте и могут влиять друг на друга.

### Гипотеза

Если оценивать `CONTINUE` и `NEW` независимо, порядок кандидатов перестанет влиять на решение.

### Эксперимент

Для каждого маршрута формируется отдельный запрос. Модель отвечает только:

- `1 = MATCH`;
- `0 = NO_MATCH`.

Для кандидата вычисляется

[
s(r)=\log P(1\mid r,x)-\log P(0\mid r,x),
]

а итоговый маршрут выбирается как кандидат с максимальным score.


In [ ]:
independent = read_json(
    "results/independent_scoring/legacy/"
    "independent_comparison_20260904_1445.json"
)

{
    "run_A_accuracy": percent(independent["run_a"]["accuracy"]),
    "run_B_accuracy": percent(independent["run_b"]["accuracy"]),
    "agreement": (
        f'{independent["semantic_agreement"]["count"]}/'
        f'{independent["semantic_agreement"]["total"]}'
    ),
    "changed_cases": len(independent["disagreements"]),
}


### Наблюдение

Гипотеза об устойчивости к порядку подтвердилась: два запуска дали agreement 30/30, то есть перестановка кандидатов не изменила ни одного решения.

Но качество оказалось низким: 16/30 = 53.33% в обоих порядках. Дополнительный анализ показал, что в 29 из 30 случаев метод предпочитал `CONTINUE`.

### Вывод

Независимая оценка устранила **position bias**, но проявила другой эффект — систематическое смещение score в пользу одного семантического кандидата.

Поэтому сама величина `logP(1)-logP(0)` также не является нейтральной мерой соответствия маршруту.

### Следующая гипотеза

Нужно проверить, связан ли этот перекос с конкретным названием `CONTINUE`, формулировкой description или вообще с использованием генеративной LLM в качестве scorer.


## 4. Диагностика представления кандидата

### Проблема

Independent scoring устойчив к порядку, но почти всегда предпочитает `CONTINUE`.

### Гипотеза

Score может зависеть от surface representation кандидата: названия маршрута или формулировки его description.

### Эксперимент

При неизменной формуле scoring проверялись варианты:

- семантические имена `CONTINUE/NEW`;
- нейтральные имена `CANDIDATE`;
- нейтральные имена `ROUTE_X`;
- эквивалентная переформулировка descriptions;
- отдельный semantic-name swap.

### Наблюдение

Численные scores менялись при замене представления, однако основной перекос в сторону `CONTINUE` сохранялся.

### Вывод

Семантический score LLM зависит от представления кандидата, но observed bias нельзя объяснить только названием маршрута.

### Следующая гипотеза

Возможно, генеративная LLM сама по себе плохо подходит для получения абсолютного relevance score. Поэтому следующий контрольный эксперимент выполняется специализированным cross-encoder reranker.


## 5. Cross-encoder reranker

### Проблема

Нужно проверить, является ли наблюдаемый bias особенностью LLM-based scoring.

### Гипотеза

Специализированный reranker должен давать более стабильное ранжирование, потому что он непосредственно обучен оценивать соответствие query и candidate text.

### Эксперимент

Reranker получает:

- query: история диалога + новое сообщение;
- два candidate texts: `CONTINUE + description` и `NEW + description`.

Никакие A/B, MATCH/NO_MATCH и token-level logprobs здесь не используются.


In [ ]:
reranker = read_json(
    "results/reranker/legacy/reranker_comparison_20260905.json"
)

{
    "run_A_accuracy": percent(reranker["run_a"]["accuracy"]),
    "run_B_accuracy": percent(reranker["run_b"]["accuracy"]),
    "agreement": (
        f'{reranker["semantic_agreement"]["count"]}/'
        f'{reranker["semantic_agreement"]["total"]}'
    ),
    "order_sensitive_cases": len(reranker["order_sensitive_cases"]),
}


### Наблюдение

Reranker оказался полностью устойчивым к перестановке кандидатов: agreement 30/30.

Однако accuracy равна 50% в обоих порядках, и все 30 случаев были классифицированы как `CONTINUE`.

### Вывод

Проблема не сводится к механизму генеративных logprobs. Общая семантическая релевантность, которую оценивает reranker, не совпадает с моей policy разделения `CONTINUE/NEW`.

Новый запрос может быть тематически очень связан с предыдущим контекстом и одновременно являться новой самостоятельной задачей.

### Следующая гипотеза

Вместо получения косвенного score нужно дать LLM саму задачу классификации и потребовать непосредственный дискретный выбор маршрута.


## 6. Direct structured routing

### Проблема

И independent scoring, и reranker пытаются сначала построить некоторую численную меру соответствия, а затем уже выбрать маршрут.

### Гипотеза

LLM может лучше решить задачу, если формулировать её непосредственно: прочитать оба определения и вернуть выбранный semantic route.

### Эксперимент

Модель получает оба маршрута и должна вернуть structured JSON с полем `route`. Token-level scoring не используется.


In [ ]:
structured = read_json(
    "results/structured_routing/legacy/"
    "structured_factorial_max128_comparison_20260907.json"
)

{
    condition: {
        "accuracy": percent(data["accuracy"]),
        "CONTINUE_predictions": data["prediction_counts"]["CONTINUE"],
        "NEW_predictions": data["prediction_counts"]["NEW"],
    }
    for condition, data in structured["conditions"].items()
}


### Наблюдение

Для route-only structured routing accuracy составляла 73.33% или 66.67% в зависимости от порядка descriptions.

При этом:

- перестановка descriptions в prompt меняла 10/30 решений;
- перестановка только порядка значений в JSON Schema enum меняла 0/30 решений.

### Вывод

Источник order effect удалось локализовать: решение зависит именно от порядка **содержательных описаний маршрутов в prompt**, а не от технического порядка enum в JSON Schema.

### Следующая гипотеза

Если перед окончательным выбором заставить модель явно сформулировать основание решения, она может меньше опираться на поверхностный порядок descriptions.


## 7. Route-only против reason + route

### Проблема

Structured route-only лучше диагностических scoring-подходов, но остаётся чувствительным к prompt order.

### Гипотеза

Явное промежуточное обоснование может заставить модель сначала обработать семантику запроса, а уже затем выбрать route.

### Эксперимент

Сравниваются два режима при одинаковых routes и модели:

- **S1: route-only** — вернуть только `route`;
- **S2: reason + route** — сначала краткий `reason`, затем `route`.

Первая попытка S2 с `max_completion_tokens=32` дала 0/120 завершённых structured responses: ответы обрывались до поля route. Я не стал извлекать решение эвристически из незавершённого JSON. Условие с лимитом 128 было проведено как отдельный эксперимент.


In [ ]:
reason = read_json(
    "results/structured_routing/legacy/"
    "structured_reason_max128_comparison_20260907.json"
)

comparison = {}
for condition in ["E1", "E2", "E3", "E4"]:
    comparison[condition] = {
        "route_only": percent(reason["s1_conditions"][condition]["accuracy"]),
        "reason_route": percent(reason["s2_conditions"][condition]["accuracy"]),
    }
comparison


### Наблюдение

После увеличения лимита до 128 S2 корректно завершался во всех случаях.

Для `reason + route` accuracy стала 27/30 = 90% во всех четырёх условиях. Число order-sensitive cases уменьшилось с 10 до 6, а agreement между противоположными prompt orders вырос с 66.67% до 80%.

При сравнении S1 → S2 на 120 condition-case комбинациях изменилось 28 решений: 26 прежних ошибок были исправлены и 2 правильных решения стали ошибочными.

### Вывод

На диагностическом наборе явное `reason` перед `route` одновременно повысило accuracy и уменьшило чувствительность к порядку descriptions. При этом инвариантность к порядку не достигнута полностью.

### Следующая гипотеза

Перед дальнейшими выводами нужно проверить, воспроизводится ли результат при повторных запусках и связан ли эффект именно с explicit reason, а не просто с включённым thinking.


## 8. Повторяемость и thinking

### Проблема

Один удачный прогон ещё не показывает, что поведение метода воспроизводимо.

### Эксперимент

По три раза были повторены три группы:

1. `reason + route`, thinking выключен;
2. `reason + route`, thinking включён;
3. `route-only`, thinking включён.

Каждый повтор содержит все условия и все 30 случаев.


In [ ]:
repeatability = read_json(
    "results/structured_routing/legacy/"
    "structured_repeatability_thinking_comparison_20260907.json"
)

# Структура файла объёмная; для просмотра можно исследовать верхние ключи.
list(repeatability.keys())


### Наблюдение

Внутри каждой группы три повтора дали одинаковые route-векторы. Для двух режимов с reason совпали даже тексты reasons.

Зафиксированные результаты одного повтора:

| Режим | Средняя accuracy | Order-sensitive cases |
|---|---:|---:|
| reason + route, thinking off | 90.00% | 6 |
| reason + route, thinking on | 88.33% | 7 |
| route-only, thinking on | 71.67% | 13 |

### Вывод

Улучшение нельзя свести к тому, что модель просто «думает дольше». В текущем setup явное поле `reason` оказалось полезнее, чем включение внутреннего thinking.

Кроме того, результат оказался детерминированно воспроизводимым в повторных запусках при зафиксированной конфигурации.

### Следующая гипотеза

После фиксации наиболее удачного pipeline можно перестать менять prompt и исследовать другую независимую переменную — размер модели.


## 9. Сравнение размера модели

### Проблема

До этого почти все выводы относились к одной модели. Неясно, является ли order sensitivity свойством конкретной модели или сохраняется при увеличении размера.

### Гипотеза

Более крупные модели могут точнее различать `CONTINUE/NEW` и быть устойчивее к перестановке descriptions.

### Эксперимент

Pipeline фиксирован:

- `reason + route`;
- thinking выключен;
- `max_completion_tokens=128`;
- два порядка descriptions: P1 и P2.

Сравниваются `Qwen3-4B-AWQ`, `Qwen3-8B-AWQ` и `Qwen3-14B-AWQ`.


In [ ]:
models = read_json(
    "results/model_comparison/legacy/"
    "model_size_20260907/comparison.json"
)

{
    label: {
        "P1": percent(data["P1"]["accuracy"]),
        "P2": percent(data["P2"]["accuracy"]),
        "mean": percent(data["mean_accuracy"]),
        "agreement": percent(data["semantic_agreement"]["agreement_rate"]),
        "order_sensitive": data["order_sensitive_case_count"],
    }
    for label, data in models["models"].items()
}


### Наблюдение

На 30-case diagnostic наборе:

| Модель | P1 | P2 | Mean | P1/P2 agreement | Sensitive cases |
|---|---:|---:|---:|---:|---:|
| 4B | 90.00% | 90.00% | 90.00% | 80.00% | 6 |
| 8B | 96.67% | 90.00% | 93.33% | 86.67% | 4 |
| 14B | 90.00% | 90.00% | 90.00% | 93.33% | 2 |

### Вывод

Рост размера модели не дал монотонного роста accuracy. На этом наборе 8B показала лучшую среднюю точность, а 14B — наибольшую устойчивость к порядку.

Поэтому «больше параметров = лучше» здесь не подтверждается простой монотонной зависимостью.

### Следующая проблема

К этому моменту один и тот же набор из 30 примеров использовался во многих экспериментах. Он уже подходит для диагностики, но не для независимой оценки итогового метода. Следующий шаг должен менять не prompt, а сам экспериментальный протокол и данные.


## 10. Переход к Dataset V3

### Причина перехода

Повторное использование 30-case набора создаёт риск исследовательского переобучения: решения о pipeline уже принимались после многократного просмотра этих примеров.

Поэтому я спроектировал новый Dataset V3 как отдельный benchmark:

- 300 случаев;
- 150 `CONTINUE` и 150 `NEW`;
- 120 development;
- 60 validation;
- 120 test;
- разные domain, category, difficulty и ambiguity;
- контрастивные семейства примеров.

Старые 30 случаев в Dataset V3 не входят.


In [ ]:
protocol = read_json(
    "results/dataset_v3_pretest/legacy/"
    "dataset_v3_pretest_20260909/FROZEN_TEST_PROTOCOL.json"
)

{
    "status": protocol["status"],
    "test_status": protocol["test_status"],
    "selected_model": protocol["selected_model_condition"]["label"],
    "validation_P1": percent(
        protocol["selection_result"]["validation_accuracy_P1"]
    ),
    "validation_P2": percent(
        protocol["selection_result"]["validation_accuracy_P2"]
    ),
    "mean_validation": percent(
        protocol["selection_result"]["mean_validation_accuracy"]
    ),
    "agreement": percent(
        protocol["selection_result"]["semantic_agreement"]
    ),
}


### Наблюдение

По заранее зафиксированному правилу после development/validation был выбран M8:

- validation P1: 81.67%;
- validation P2: 76.67%;
- mean validation accuracy: 79.17%;
- agreement: 91.67%.

При этом простой message-only TF-IDF baseline достиг 81.67% на validation. Это является тревожным признаком: в синтетическом Dataset V3 может присутствовать surface leakage, позволяющий угадывать класс без полноценного использования истории диалога.

### Вывод

На этом этапе корректный следующий шаг — не открывать test и не продолжать подбор prompt. Сначала необходимо проверить качество разметки и наличие lexical/surface leakage.

Test в зафиксированном протоколе не запускался.


## Итоговая последовательность исследования

Вся цепочка экспериментов получилась не как набор независимых попыток, а как последовательное уточнение причины ошибки:

**A/B logprobs**  
→ обнаружилась зависимость от представления классов

**Семантические имена и перестановка кандидатов**  
→ выяснилось, что зависимость сохраняется без A/B

**Independent candidate scoring**  
→ порядок перестал влиять, но появился сильный bias в сторону CONTINUE

**Representation diagnostics**  
→ score зависит от surface representation, но этим bias полностью не объясняется

**Cross-encoder reranker**  
→ порядок устойчив, но модель полностью схлопывается в CONTINUE

**Direct structured routing**  
→ качество выше, однако снова появляется order sensitivity

**Факторная проверка prompt order / enum order**  
→ источник эффекта локализован в порядке descriptions, а не JSON enum

**Reason + route**  
→ accuracy выросла до 90%, order sensitivity уменьшилась

**Повторяемость и thinking**  
→ эффект воспроизводится; thinking сам по себе улучшения не объясняет

**Сравнение 4B / 8B / 14B**  
→ размер влияет на точность и устойчивость по-разному, монотонной зависимости нет

**Dataset V3**  
→ переход от диагностического набора к более строгому validation/test протоколу; обнаружен риск surface leakage, поэтому test остаётся закрытым.

Таким образом, исследование постепенно сместилось от вопроса **«как получить вероятность намерения из logprobs?»** к более точному вопросу: **«какой способ представления и выбора семантического маршрута даёт одновременно приемлемую точность и устойчивость к нерелевантным изменениям prompt?»**
